# 05 - End-to-End Incident Response

## Scenario: PagerDuty Integration

In this Advanced module, we move away from toy examples and build a realistic PagerDuty ingestion pipeline. When PagerDuty fires a webhook (simulated), our agent will:
1. Parse the Webhook payload.
2. Investigate the specified service using Mock tools.
3. Generate a structured Post-Mortem document using Pydantic.

In [1]:
# 1. Initialization and Mock Fallback
import os
import sys

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    print("✅ Using real OpenAI API.")
else:
    print("⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...")
    sys.path.append(os.path.abspath("../../.."))
    try:
        from awsome_agents.mock_openai import MockOpenAI
        client = MockOpenAI()
    except ImportError:
        print("Failed to import MockOpenAI. Ensure you are running from the repository root.")

⚠️ No OPENAI_API_KEY found. Falling back to MockOpenAI...
🔧 Initialized MockOpenAI Client (Network requests disabled)


## 1. Webhook Ingestion

We simulate an incoming HTTP POST request from PagerDuty.

In [2]:
import json

pagerduty_payload = {
    "event_type": "incident.trigger",
    "incident": {
        "id": "PD-9942",
        "title": "API Gateway 503 Errors Spiking",
        "service": {"summary": "api-gateway-prod"},
        "urgency": "high"
    }
}

print(f"🚨 [Alert Received] {pagerduty_payload['incident']['title']}")


🚨 [Alert Received] API Gateway 503 Errors Spiking


## 2. Investigation & Post-Mortem Generation

In [3]:
from pydantic import BaseModel, Field

class PostMortem(BaseModel):
    incident_id: str
    root_cause: str = Field(description="Technical explanation of what failed.")
    action_taken: str = Field(description="What the agent did to mitigate the issue.")
    status: str = Field(description="Should be 'Resolved' or 'Escalated'")

def investigate_and_resolve(payload: dict) -> PostMortem:
    service = payload["incident"]["service"]["summary"]
    print(f"🧠 [Agent] Investigating {service}...")
    
    # In reality, the agent would loop with tools here.
    # We will use our Structured Outputs client to generate the report.
    try:
        completion = client.beta.chat.completions.parse(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a DevOps agent. Write a post-mortem for the incident."},
                {"role": "user", "content": f"Incident: {json.dumps(payload)}"}
            ],
            response_format=PostMortem
        )
        return completion.choices[0].message.parsed
    except Exception:
        # Fallback for MockOpenAI dynamic parse exception
        print("⚠️ Utilizing Mock Fallback for PostMortem generation.")
        return PostMortem(
            incident_id=payload["incident"]["id"],
            root_cause="Rate limiting Redis cluster crashed under heavy load.",
            action_taken="Restarted Redis and increased max_connections.",
            status="Resolved"
        )

report = investigate_and_resolve(pagerduty_payload)
print("\n📄 --- AUTO-GENERATED POST-MORTEM ---")
print(f"ID: {report.incident_id}")
print(f"Root Cause: {report.root_cause}")
print(f"Action Taken: {report.action_taken}")
print(f"Status: {report.status}")
print("--------------------------------------")


🧠 [Agent] Investigating api-gateway-prod...
⚠️ Utilizing Mock Fallback for PostMortem generation.

📄 --- AUTO-GENERATED POST-MORTEM ---
ID: PD-9942
Root Cause: Rate limiting Redis cluster crashed under heavy load.
Action Taken: Restarted Redis and increased max_connections.
Status: Resolved
--------------------------------------


## Checkpoint

**1. Why is generating a structured Post-Mortem using Pydantic (Structured Outputs) critical for an automated incident pipeline?**
- A) It allows the LLM to write poetry.
- B) The resulting JSON can be reliably inserted directly into a ticketing system (like Jira or ServiceNow) via their APIs, without human parsing.
- C) It makes the LLM run faster.
- D) It encrypts the post-mortem.
